# Real Madrid AI Transfer Intelligence
## Notebook 02 — Data Cleaning & Quality

**Goal:** clean the player-season extract from Notebook 01, standardize columns, validate data quality, calculate per-90 metrics and export an analysis-ready dataset.

**Input:** `data/processed/real_madrid_player_season_laliga_2023_2026.csv`

No missing statistics are converted to zero. Conflicting duplicate player-season records are reported, not silently deleted.

## 1. Load data

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np
import re, json, unicodedata
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = DATA_DIR / "real_madrid_player_season_laliga_2023_2026.csv"
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Missing {INPUT_PATH}. Run Notebook 01 first.")

raw = pd.read_csv(INPUT_PATH)
print("Shape:", raw.shape)
print("Columns:", raw.columns.tolist())
display(raw.head())


,league,season,team,player_name,nation,pos,age,born,playing_time_mp,playing_time_starts,...,performance_cs,performance_cs%,penalty_kicks_pkatt,penalty_kicks_pka,penalty_kicks_pksv,penalty_kicks_pkm,penalty_kicks_save%,source,retrieved_at_utc,scope
0,ESP-La Liga,2324,Real Madrid,Andriy Lunin,UKR,GK,24,1999,21,21,...,4.0,19.0,1.0,1.0,0.0,0.0,0.0,FBref via soccerdata,2026-11-05T08:28:00.400181+00:00,La Liga
1,ESP-La Liga,2324,Real Madrid,Antonio Rüdiger,GER,DF,30,1993,33,30,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FBref via soccerdata,2026-11-05T08:28:00.400181+00:00,La Liga
2,ESP-La Liga,2324,Real Madrid,Arda Güler,TUR,"MF,FW",18,2005,10,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FBref via soccerdata,2026-11-05T08:28:00.400181+00:00,La Liga
3,ESP-La Liga,2324,Real Madrid,Aurélien Tchouaméni,FRA,"MF,DF",23,2000,27,22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FBref via soccerdata,2026-11-05T08:28:00.400181+00:00,La Liga
4,ESP-La Liga,2324,Real Madrid,Brahim Díaz,MAR,"MF,FW",23,1999,31,18,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,FBref via soccerdata,2026-11-05T08:28:00.400181+00:00,La Liga


## 2. Normalize column names and map key fields

In [7]:
def norm_col(c):
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(c).strip().lower()
    ).strip("_")


# Нормализуем названия
normalized_names = [norm_col(c) for c in raw.columns]

# Группируем одинаковые названия по позициям
column_positions = {}

for position, name in enumerate(normalized_names):
    column_positions.setdefault(name, []).append(position)


# Создаём новый DataFrame
# с гарантированно уникальными названиями
cleaned_columns = {}
duplicate_report = []

for name, positions in column_positions.items():

    duplicated_data = raw.iloc[:, positions].copy()

    # Считаем строки, где значения конфликтуют
    conflicts = (
        duplicated_data.nunique(axis=1, dropna=True) > 1
    )

    duplicate_report.append({
        "column": name,
        "original_columns": len(positions),
        "conflicting_rows": int(conflicts.sum())
    })

    # Объединяем непустые значения
    cleaned_columns[name] = (
        duplicated_data.bfill(axis=1).iloc[:, 0]
    )


# Пересоздаём таблицу
df = pd.DataFrame(cleaned_columns, index=raw.index)

# Проверяем
print("Original shape:", raw.shape)
print("New shape:", df.shape)
print("Columns unique:", df.columns.is_unique)

display(pd.DataFrame(duplicate_report))

assert df.columns.is_unique

print("SUCCESS: duplicate column names removed!")

SUCCESS: duplicate column names removed!


In [8]:
candidates = {
    "player_name": ["player_name", "player", "name"],
    "season": ["season", "year"],
    "team": ["team", "squad", "team_name", "squad_name"],
    "competition": ["competition", "comp", "league"],
    "position": ["position", "pos"],
    "age": ["age"],
    "minutes": ["minutes", "min", "playing_time_min"],
    "matches": ["matches", "mp", "appearances"],
    "starts": ["starts", "playing_time_starts"],
    "goals": ["goals", "gls", "standard_gls"],
    "assists": ["assists", "ast", "standard_ast"],
}

column_map = {
    key: next(
        (c for c in values if c in df.columns),
        None
    )
    for key, values in candidates.items()
}

display(pd.DataFrame([
    {"field": k, "source_column": v}
    for k, v in column_map.items()
]))

missing_keys = [
    key for key in ["player_name", "season"]
    if column_map[key] is None
]

if missing_keys:
    raise ValueError(
        f"Missing fields: {missing_keys}\n"
        f"Available: {df.columns.tolist()}"
    )

for target, source in column_map.items():
    if source is not None and target not in df.columns:
        df[target] = df[source]

print("Field mapping completed!")

Field mapping completed!


## 3. Standardize values and create player IDs

In [9]:
for col in ["player_name", "season", "team", "competition", "position"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

df["season"] = df["season"].str.replace("/", "-", regex=False).str.replace("_", "-", regex=False)

for col in ["age", "minutes", "matches", "starts", "goals", "assists"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

def make_player_id(name):
    if pd.isna(name):
        return pd.NA
    name = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

df["player_id"] = df["player_name"].map(make_player_id).astype("string")
if "team" in df.columns:
    df["club"] = df["team"]

display(df[[c for c in ["player_id", "player_name", "season", "club", "position", "minutes", "goals", "assists"] if c in df.columns]].head(12))


,player_id,player_name,season,club,position,minutes,goals
0,andriy_lunin,Andriy Lunin,2324,Real Madrid,GK,1890.0,0
1,antonio_rudiger,Antonio Rüdiger,2324,Real Madrid,DF,2709.0,1
2,arda_guler,Arda Güler,2324,Real Madrid,"MF,FW",377.0,6
3,aurelien_tchouameni,Aurélien Tchouaméni,2324,Real Madrid,"MF,DF",1987.0,3
4,brahim_diaz,Brahim Díaz,2324,Real Madrid,"MF,FW",1545.0,8
5,dani_carvajal,Dani Carvajal,2324,Real Madrid,DF,2171.0,4
6,dani_ceballos,Dani Ceballos,2324,Real Madrid,MF,598.0,0
7,david_alaba,David Alaba,2324,Real Madrid,DF,1154.0,0
8,eduardo_camavinga,Eduardo Camavinga,2324,Real Madrid,MF,1817.0,0
9,federico_valverde,Federico Valverde,2324,Real Madrid,MF,2904.0,2


## 4. Check duplicate player-season keys

In [10]:
key_cols = ["player_id", "season"]
for col in ["club", "competition"]:
    if col in df.columns:
        key_cols.append(col)

exact_dupes = int(df.duplicated().sum())
df_clean = df.drop_duplicates().copy()
duplicate_mask = df_clean.duplicated(key_cols, keep=False)
duplicate_review = df_clean.loc[duplicate_mask].sort_values(key_cols).copy()

print("Key columns:", key_cols)
print("Exact duplicate rows removed:", exact_dupes)
print("Rows in unresolved duplicate keys:", len(duplicate_review))
if len(duplicate_review):
    display(duplicate_review.head(30))


Key columns: ['player_id', 'season', 'club', 'competition']
Exact duplicate rows removed: 0
Rows in unresolved duplicate keys: 0


## 5. Validate numeric ranges

In [11]:
checks = []

def add_check(name, mask, rule):
    count = int(mask.fillna(False).sum())
    checks.append({"check": name, "flagged_rows": count, "rule": rule})
    if count:
        display(df_clean.loc[mask.fillna(False)].head(20))

if "minutes" in df_clean:
    add_check("negative_minutes", df_clean["minutes"] < 0, "minutes >= 0")
if "goals" in df_clean:
    add_check("negative_goals", df_clean["goals"] < 0, "goals >= 0")
if "assists" in df_clean:
    add_check("negative_assists", df_clean["assists"] < 0, "assists >= 0")
if "age" in df_clean:
    add_check("age_outside_15_45", (df_clean["age"] < 15) | (df_clean["age"] > 45), "review ages outside 15–45")
if {"starts", "matches"}.issubset(df_clean.columns):
    add_check("starts_exceed_matches", df_clean["starts"] > df_clean["matches"], "starts <= matches")

checks_df = pd.DataFrame(checks)
display(checks_df)


,check,flagged_rows,rule
0,negative_minutes,0,minutes >= 0
1,negative_goals,0,goals >= 0
2,age_outside_15_45,0,review ages outside 15–45


## 6. Missingness analysis

In [12]:
missingness = (
    df_clean.isna().mean().mul(100)
    .sort_values(ascending=False)
    .rename("missing_percent").to_frame()
)
display(missingness.round(1))

if "season" in df_clean.columns:
    season_quality = df_clean.groupby("season", dropna=False).agg(
        rows=("player_id", "size"),
        unique_players=("player_id", "nunique"),
        missing_player_id=("player_id", lambda s: int(s.isna().sum()))
    ).reset_index()
    display(season_quality)


,season,rows,unique_players,missing_player_id
0,2324,27,27,0
1,2425,30,30,0
2,2526,35,35,0


## 7. Engineer per-90 metrics

In [13]:
if "minutes" in df_clean.columns:
    valid_minutes = df_clean["minutes"].gt(0)
    for metric in ["goals", "assists"]:
        if metric in df_clean.columns:
            df_clean[f"{metric}_per_90"] = np.where(
                valid_minutes, df_clean[metric] * 90 / df_clean["minutes"], np.nan
            )

if {"goals", "assists"}.issubset(df_clean.columns):
    df_clean["goal_contributions"] = df_clean["goals"] + df_clean["assists"]
    if "minutes" in df_clean.columns:
        df_clean["goal_contributions_per_90"] = np.where(
            df_clean["minutes"].gt(0),
            df_clean["goal_contributions"] * 90 / df_clean["minutes"],
            np.nan
        )

display(df_clean[[c for c in ["player_name", "season", "minutes", "goals", "assists", "goals_per_90", "assists_per_90", "goal_contributions_per_90"] if c in df_clean.columns]].head(15))


,player_name,season,minutes,goals,goals_per_90
0,Andriy Lunin,2324,1890.0,0,0.000000
1,Antonio Rüdiger,2324,2709.0,1,0.033223
2,Arda Güler,2324,377.0,6,1.432361
3,Aurélien Tchouaméni,2324,1987.0,3,0.135883
4,Brahim Díaz,2324,1545.0,8,0.466019
5,Dani Carvajal,2324,2171.0,4,0.165822
6,Dani Ceballos,2324,598.0,0,0.000000
7,David Alaba,2324,1154.0,0,0.000000
8,Eduardo Camavinga,2324,1817.0,0,0.000000
9,Federico Valverde,2324,2904.0,2,0.061983


## 8. Export cleaned data and quality reports

In [14]:
clean_path = DATA_DIR / "real_madrid_player_season_clean.csv"
players_path = DATA_DIR / "real_madrid_players_clean.csv"
duplicate_path = REPORTS_DIR / "duplicate_key_review.csv"
missing_path = REPORTS_DIR / "missingness_by_column.csv"
range_path = REPORTS_DIR / "range_check_report.csv"
summary_path = REPORTS_DIR / "data_quality_summary.json"

df_clean.to_csv(clean_path, index=False)
df_clean[["player_id", "player_name"]].drop_duplicates().sort_values("player_name").to_csv(players_path, index=False)
duplicate_review.to_csv(duplicate_path, index=False)
missingness.to_csv(missing_path)
checks_df.to_csv(range_path, index=False)

summary = {
    "input_rows": int(len(raw)),
    "cleaned_rows": int(len(df_clean)),
    "unique_players": int(df_clean["player_id"].nunique()),
    "exact_duplicate_rows_removed": exact_dupes,
    "unresolved_duplicate_key_rows": int(len(duplicate_review)),
    "missing_player_ids": int(df_clean["player_id"].isna().sum()),
    "key_columns": key_cols,
    "range_checks": checks,
    "status": "PASS_WITH_REVIEW" if len(duplicate_review) == 0 and not any(x["flagged_rows"] for x in checks) else "REVIEW_REQUIRED",
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2, ensure_ascii=False))


{
  "input_rows": 92,
  "cleaned_rows": 92,
  "unique_players": 50,
  "exact_duplicate_rows_removed": 0,
  "unresolved_duplicate_key_rows": 0,
  "missing_player_ids": 0,
  "key_columns": [
    "player_id",
    "season",
    "club",
    "competition"
  ],
  "range_checks": [
    {
      "check": "negative_minutes",
      "flagged_rows": 0,
      "rule": "minutes >= 0"
    },
    {
      "check": "negative_goals",
      "flagged_rows": 0,
      "rule": "goals >= 0"
    },
    {
      "check": "age_outside_15_45",
      "flagged_rows": 0,
      "rule": "review ages outside 15–45"
    }
  ],
  "status": "PASS_WITH_REVIEW",
  "created_at_utc": "2026-11-05T09:40:02.848289+00:00"
}
